# MCP: a shared way to expose tools

MCP lets a program discover and call tools from another process.

In this notebook, we will:

- build the small JSON-RPC messages by hand;
- initialize a connection;
- list and call tools;
- separate tool failures from protocol failures;
- use the official MCP SDK; and
- adapt MCP tools to our existing agent loop.

Before running the SDK section, install the MCP extra once with: pip install -e ".[mcp]".

In [ ]:
import asyncio
import json
import os
import sys
import threading
from getpass import getpass
from pathlib import Path

import litellm
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent
load_dotenv(course_root / ".env")
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
litellm.drop_params = True
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=str(course_root / ".litellm_cache"))

## 1. MCP uses JSON-RPC messages

JSON-RPC gives each request an id, method name, and parameters. A response contains either result or error.

In [ ]:
def request(method, params=None, request_id=1):
    return {
        "jsonrpc": "2.0",
        "id": request_id,
        "method": method,
        "params": params or {},
    }

def success(request_id, result):
    return {"jsonrpc": "2.0", "id": request_id, "result": result}

def failure(request_id, code, message):
    return {
        "jsonrpc": "2.0",
        "id": request_id,
        "error": {"code": code, "message": message},
    }

print(json.dumps(request("tools/list"), indent=2))

We will make a tiny in-memory server with two tools.

In [ ]:
def refund_preview(order_id, amount_usd):
    order_total = 391.50
    if amount_usd > order_total:
        raise ValueError("refund is larger than the order total")
    return {"order_id": order_id, "amount_usd": amount_usd, "allowed": True}

TOY_TOOLS = {
    "get_order": {
        "schema": {"type": "object", "properties": {"order_id": {"type": "string"}}},
        "function": lambda order_id: {"order_id": order_id, "status": "delivered", "total_usd": 391.50},
    },
    "refund_preview": {
        "schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "amount_usd": {"type": "number"},
            },
        },
        "function": refund_preview,
    },
}

def tool_result(value, is_error=False):
    text = value if isinstance(value, str) else json.dumps(value)
    return {"content": [{"type": "text", "text": text}], "isError": is_error}

def handle_tool_call(params):
    tool = TOY_TOOLS.get(params["name"])
    if tool is None:
        return tool_result("unknown tool: " + params["name"], is_error=True)
    try:
        return tool_result(tool["function"](**params.get("arguments", {})))
    except Exception as error:
        return tool_result(str(error), is_error=True)

In [ ]:
def handle(message):
    method = message["method"]
    request_id = message["id"]

    if method == "initialize":
        return success(request_id, {
            "protocolVersion": "2025-11-25",
            "serverInfo": {"name": "toy-shop", "version": "0.1"},
            "capabilities": {"tools": {}},
        })

    if method == "tools/list":
        tools = [
            {"name": name, "inputSchema": value["schema"]}
            for name, value in TOY_TOOLS.items()
        ]
        return success(request_id, {"tools": tools})

    if method == "tools/call":
        return success(request_id, handle_tool_call(message["params"]))

    return failure(request_id, -32601, "method not found: " + method)

## 2. Initialize, list, call

The client first asks which protocol version and capabilities the server supports.

In [ ]:
hello = handle(request("initialize", {
    "protocolVersion": "2025-11-25",
    "clientInfo": {"name": "learning-client", "version": "0.1"},
}))

print(hello["result"])

In [ ]:
listed = handle(request("tools/list"))["result"]["tools"]
print("tools:", [tool["name"] for tool in listed])

called = handle(request("tools/call", {
    "name": "get_order",
    "arguments": {"order_id": "ORD-7301"},
}))
print(called["result"])

## 3. Separate two kinds of failure

If tools/call runs but the tool fails, the response is still a normal result with isError set to true.

If the server does not know the JSON-RPC method, the response has an error object.

In [ ]:
bad_tool_input = handle(request("tools/call", {
    "name": "refund_preview",
    "arguments": {"order_id": "ORD-7301", "amount_usd": 999999},
}))
unknown_method = handle(request("not/a-real-method"))

print("tool failure:", bad_tool_input["result"])
print("protocol failure:", unknown_method["error"])

## 4. Use the official SDK

The SDK manages the same handshake and messages. The course uses MCP 1.x because the surrounding agent libraries currently depend on that major version.

This example starts the course's shop-desk server as a child process. The context managers close it when the cell finishes.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER = course_root / "mcp_servers/shopdesk_server.py"

async def explore_server():
    params = StdioServerParameters(
        command=sys.executable,
        args=[str(SERVER)],
        env=os.environ.copy(),
    )

    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            hello = await session.initialize()
            tools = await session.list_tools()
            order = await session.call_tool("get_order", {"order_id": "ORD-7301"})

            print("server:", hello.serverInfo.name)
            print("tools:", [tool.name for tool in tools.tools])
            print("order result:", order.content[0].text)

await explore_server()

MCP has three main primitives:

- tools perform actions;
- resources provide readable data; and
- prompts provide reusable message templates.

## 5. Adapt MCP tools to our agent loop

Our loop expects shoplab Tool objects. An adapter can keep the tool name, description, and schema while sending the real call through MCP.

To keep the adapter easy to read, this learning version opens one short MCP session per tool call.

In [ ]:
def run_async_in_thread(async_function):
    box = {}

    def worker():
        try:
            box["value"] = asyncio.run(async_function())
        except Exception as error:
            box["error"] = error

    thread = threading.Thread(target=worker)
    thread.start()
    thread.join()

    if "error" in box:
        raise box["error"]
    return box["value"]

In [ ]:
from shoplab.tools import Tool, standard_tools

def server_params():
    return StdioServerParameters(
        command=sys.executable,
        args=[str(SERVER)],
        env=os.environ.copy(),
    )

async def list_remote_tools():
    async with stdio_client(server_params()) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            await session.initialize()
            return (await session.list_tools()).tools

async def call_remote_tool(name, arguments):
    async with stdio_client(server_params()) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            await session.initialize()
            result = await session.call_tool(name, arguments)
            if result.isError:
                return {"error": result.content[0].text}
            return json.loads(result.content[0].text)

remote_specs = run_async_in_thread(list_remote_tools)
remote_tools = {}

for spec in remote_specs:
    def remote_function(_name=spec.name, **arguments):
        return run_async_in_thread(lambda: call_remote_tool(_name, arguments))

    remote_tools[spec.name] = Tool(
        spec.name,
        spec.description or "MCP tool",
        spec.inputSchema,
        remote_function,
    )

remote_tools["finish"] = standard_tools()["finish"]
print(list(remote_tools))

In [ ]:
from shoplab.loop import run_agent

task = (
    "Triage the opened Torrent boots on order ORD-7312. "
    "Look up the order and customer, search the policy once, then call finish "
    "with decision, policy_id, and refund_usd."
)

result = run_agent(
    task,
    remote_tools,
    model=MODEL,
    system="Use the MCP-backed facts. Do not repeat lookups.",
    max_steps=8,
)

print("stop:", result.stop_reason)
print("answer:", result.answer)

The agent loop did not change. Only the tool transport changed. Local Python tools and MCP tools can share the same loop interface.

## What to remember

1. MCP uses JSON-RPC with agreed method names.
2. initialize starts the relationship.
3. tools/list supports discovery.
4. tools/call returns content and an isError flag.
5. Protocol errors are different from tool errors.
6. An adapter lets the existing agent loop use MCP tools.

## Small practice steps

1. Add a new tool to TOY_TOOLS.
2. Call a missing toy tool and inspect isError.
3. Print the schema of one remote MCP tool.